In [1]:
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

In [5]:
import torch
import torch.nn as nn
import torch.optim as optim

import torchvision  # PtTorch libraray mainly used for Computer vision
from torchvision.datasets import CIFAR10


In [13]:
# Datsets and DataLoader

from torch.utils.data import DataLoader
import torchvision.transforms as transforms

# image(0,255) - scale(0,1) - Normalize(-1,1)
# Define transformer for which each image two operation perform(scaling & normalize)

transform = transforms.Compose([  # compose function each image multiple layer(scale , noramlize) perform

     transforms.ToTensor(), # ToTensor - data tensor convert and simutaneously scaling perform
     transforms.Normalize( (0.5, 0.5, 0.5) , (0.5, 0.5 , 0.5)) # Normalize operation perform & values by deafult set

    
])


trainset = CIFAR10(root = "./data1" , train = True ,  download =True , transform = transform)
testset = CIFAR10(root = "./data1" , train = False , download = True , transform = transform)

100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 170M/170M [38:15<00:00, 74.3kB/s]


In [14]:
# DataLoader - define how the data will be loaded for traning
train_loader =  DataLoader(trainset , batch_size = 64 , shuffle = True)
test_loader = DataLoader(testset , batch_size = 64)

### Build CNN

In [29]:
class CNN(nn.Module): # class CNN - inherit functions & attributes from base class (nn.Module) - Logic of Pytorch is written
    def __init__(self): # constructor define for CNN class , self refers to the object call
        super(CNN , self).__init__() # base constructor Module class,  call

        self.conv_layers = nn.Sequential( # conv_layers - attribute of class CNN

           nn.Conv2d(3 , 32 , kernel_size = 3, padding = 1), # (3 - Input Channels ) (32 - o/p Feature map generate or we can say 32 filter/kernel apply)
           nn.ReLU(),
           nn.MaxPool2d(2,2), # Kernel_size = 2*2 , stride = 2

           nn.Conv2d(32, 64, kernel_size = 3, padding =1), # (32 - Input Channels ) (64 - o/p Feature map generate or we can say 64 filter/kernel apply)
           nn.ReLU(),
           nn.MaxPool2d(2,2),

           nn.Conv2d(64, 128, kernel_size = 3, padding =1), # (64 - Input Channels ) (128 - o/p Feature map generate or we can say 128 filter/kernel apply)
           nn.ReLU(),
           nn.MaxPool2d(2,2)   #(o/p genrate size - 4*4*128)

        )

        self.fc_layers = nn.Sequential(

            # hidden layer define 
            # nn.Linear(i/p neurons , o/p neurons)
            nn.Linear(4*4*128 , 256),  #(256 - random o/p value)
            nn.ReLU(),

            nn.Linear(256, 10)  # Last o/p layer Softmax AF apply (Loss fnx by default use softmax AF so here simply Linaer AF apply)
             # o/p 10 classes
            
        )

            
            
    def forward(self , x):
       x = self.conv_layers(x) # x input image pass conv_layer
       x = x.view(x.size(0) , -1) # flattening o/p size 4*4*128
       x = self.fc_layers(x) # flatten size pass fc_layers o/p 10 classes

       return x
            
    
        

In [30]:
model = CNN()    # object create of call CNN, constructor call

In [31]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [32]:
print(model)

CNN(
  (conv_layers): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc_layers): Sequential(
    (0): Linear(in_features=2048, out_features=256, bias=True)
    (1): ReLU()
    (2): Linear(in_features=256, out_features=10, bias=True)
  )
)


In [34]:
#print(list(model.parameters()))

### Train CNN

In [36]:
epochs = 10

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:

        optimizer.zero_grad()  # for each batch fresh gradient compute, dont accumualte value of graident of previous batch
        output = model.forward(images) # Forward Propgation
        loss = criterion(output, labels) # Loss compute
        loss.backward() # Backward propgation , Gradient compute automatically
        optimizer.step() # learn params update

        running_loss += loss.item()  # tensor valur covert python float vlaue
                                     # Runnig_loss cal for 782 batches for 1 epoch

    train_loss = running_loss/782    # Avg loss per batch for 1 epoch
        
    print(f"epoch = {epoch}/{epochs} , loss = {train_loss}")   
        
        

    
    
# model store best laern parameters , now our model ready for predictions

epoch = 0/10 , loss = 1.4108537941637551
epoch = 1/10 , loss = 0.9661090352651104
epoch = 2/10 , loss = 0.7845322237447705
epoch = 3/10 , loss = 0.6627642919721506
epoch = 4/10 , loss = 0.558039834546616
epoch = 5/10 , loss = 0.4744423783724875
epoch = 6/10 , loss = 0.39425829132956924
epoch = 7/10 , loss = 0.32366170462154215
epoch = 8/10 , loss = 0.25779279127069143
epoch = 9/10 , loss = 0.20464765858810272


In [35]:
len(train_loader) # No of batches

782

### Evaluation

In [37]:
model.eval()

correct =0

with torch.no_grad(): # No automatically gradient compute

    for images , labels in test_loader:

        output = model.forward(images) # FP , here model contains best learn params , min loss value
        _,pred = torch.max(output , 1) # torch.max - return max value and index
          # here pred = index
        # o/p  - 10 values [1, 0.6, 1.4 , 0.5 , -0.2 , 1.2, 0.1  , 3 , 2 ,1]
        #( max value = 3 , index = 7)
        correct += (pred == labels).sum().item()

        

        

        
    

    
    

In [39]:
print(labels.shape)

torch.Size([16])


In [40]:
labels.shape

torch.Size([16])

In [43]:
len(testset)

10000

In [44]:
print(correct)

7515


In [45]:
# Accuracy

print(correct/len(testset)*100)

75.14999999999999


In [46]:
correct

7515